# Energy Consumption Prediction

**Regression Project 56**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict the heating load of buildings based on architectural and structural parameters.
The UCI Energy Efficiency dataset contains 768 simulated building shapes analysed using
Ecotect software, with 8 input variables describing geometry and glazing, and 2 output
variables (Heating Load and Cooling Load).

We focus on **Heating Load** as the primary regression target — a practical task for
energy-efficient building design.

## 3. Learning Objectives

1. Build a regression pipeline for energy performance prediction
2. Work with a clean, well-documented UCI dataset
3. Understand how building geometry affects energy consumption
4. Compare linear models vs tree-based models on structured numeric data
5. Evaluate with R², RMSE, MAE and residual analysis
6. Practice the full LazyPredict → FLAML → top-3 workflow

## 4. Problem Statement

> Given 8 building parameters (compactness, surface area, wall area, roof area, height,
> orientation, glazing area, glazing distribution), predict the **Heating Load** (in kWh/m²).

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Architects | Design energy-efficient buildings |
| Building engineers | HVAC system sizing |
| Energy auditors | Assess building performance |
| Policy makers | Building energy codes |
| ML learners | Clean numeric regression with domain meaning |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `elikplim/uci-energy-efficiency-dataset` |
| Records | 768 |
| Features | 8 numeric building parameters |
| Targets | Heating Load, Cooling Load (we use Heating Load) |
| Key columns | Relative_Compactness, Surface_Area, Wall_Area, Roof_Area, Overall_Height, Orientation, Glazing_Area, Glazing_Area_Distribution |

## 7. Dataset Source and License Notes

- **Kaggle**: [elikplim/uci-energy-efficiency-dataset](https://www.kaggle.com/datasets/elikplim/uci-energy-efficiency-dataset)
- Originally from UCI ML Repository (Tsanas & Xifara, 2012)
- License: CC0 / Public Domain
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'elikplim/uci-energy-efficiency-dataset'
TARGET = 'Y1'  # Heating Load — column name in the dataset
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 90
MAX_ROWS = 50000

## 11. Dataset Download or Loading

We download the UCI Energy Efficiency dataset via `kagglehub`. The dataset columns are
labelled X1–X8 for features and Y1/Y2 for targets.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/elikplim/uci-energy-efficiency-dataset'
    )
all_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
csv_files = [f for f in all_files if f.endswith('.csv')]
xlsx_files = [f for f in all_files if f.endswith('.xlsx')]
if csv_files:
    df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
elif xlsx_files:
    df_raw = pd.read_excel(sorted(xlsx_files, key=os.path.getsize, reverse=True)[0])
else:
    raise FileNotFoundError(f'No CSV/XLSX files in {path}: {all_files}')
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Verify the target column, rename columns for readability, and check data quality.

In [ ]:
# The dataset may have columns X1-X8, Y1, Y2 or descriptive names
col_map = {
    'X1': 'Relative_Compactness', 'X2': 'Surface_Area', 'X3': 'Wall_Area',
    'X4': 'Roof_Area', 'X5': 'Overall_Height', 'X6': 'Orientation',
    'X7': 'Glazing_Area', 'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load', 'Y2': 'Cooling_Load'
}
df = df_raw.copy()
df = df.rename(columns=col_map)
TARGET = 'Heating_Load'
if TARGET not in df.columns:
    # Try original name
    TARGET = 'Y1'
assert TARGET in df.columns, f"Target '{TARGET}' not found in {list(df.columns)}"
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'Duplicated rows: {df.duplicated().sum()}')
# Drop Cooling_Load — we focus on Heating_Load
drop_cols = [c for c in ['Cooling_Load', 'Y2'] if c in df.columns]
if drop_cols: df = df.drop(columns=drop_cols)
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
print(f'Shape: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

All 8 features are numeric, making this a clean dataset for EDA.

In [ ]:
df.describe()

In [ ]:
num_feats = [c for c in df.columns if c != TARGET]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, num_feats[:8]):
    df[col].hist(bins=25, ax=ax, alpha=0.7, color='steelblue')
    ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

In [ ]:
# Top feature correlations with target
top_corr = df.corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, top_corr.head(4).index):
    ax.scatter(df[col], df[TARGET], alpha=0.4, s=15)
    ax.set_xlabel(col); ax.set_ylabel(TARGET)
    ax.set_title(f'{col} vs {TARGET}')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Heating Load distribution — bimodal pattern is expected due to different building heights.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=40, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution')
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title(f'{TARGET} Box Plot')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():.2f} kWh/m\u00b2')
print(f'Median: {df[TARGET].median():.2f}')
print(f'Std: {df[TARGET].std():.2f}')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split with fixed seed. Dataset is small (768 rows).

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- All features are numeric — no categorical encoding needed
- Impute with median (for robustness) and scale with StandardScaler
- Scaling helps linear models; tree models are invariant

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Numeric features: {len(num_cols)}')

## 17. Feature Engineering

We create ratio-based features that may capture building efficiency better:
- **Wall-to-floor ratio**: Wall_Area / Roof_Area
- **Glazing-to-wall ratio**: Glazing_Area / Wall_Area
- **Volume proxy**: Surface_Area × Overall_Height

In [ ]:
def engineer_features(d):
    d = d.copy()
    cols = d.columns.tolist()
    wa = 'Wall_Area' if 'Wall_Area' in cols else 'X3'
    ra = 'Roof_Area' if 'Roof_Area' in cols else 'X4'
    ga = 'Glazing_Area' if 'Glazing_Area' in cols else 'X7'
    sa = 'Surface_Area' if 'Surface_Area' in cols else 'X2'
    oh = 'Overall_Height' if 'Overall_Height' in cols else 'X5'
    if wa in cols and ra in cols:
        d['wall_floor_ratio'] = d[wa] / d[ra].replace(0, np.nan)
        d['wall_floor_ratio'] = d['wall_floor_ratio'].fillna(0)
    if ga in cols and wa in cols:
        d['glazing_wall_ratio'] = d[ga] / d[wa].replace(0, np.nan)
        d['glazing_wall_ratio'] = d['glazing_wall_ratio'].fillna(0)
    if sa in cols and oh in cols:
        d['volume_proxy'] = d[sa] * d[oh]
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)
], remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor, LinearRegression, RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:.2f}, MAE={r['MAE']:.2f}")

## 19. LazyPredict Benchmark

Quick comparison of ~30 regression models.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML with 90-second budget.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:.2f}")

## 21. Top 3 Model Selection

Based on benchmarks, we select three strong models.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on training set, evaluate on held-out test set.

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:.2f}, MAE={final[name]['MAE']:.2f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.5, s=15)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.5, s=15)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Inspect the best model's largest errors.

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():.2f}')
print(f'Median absolute error: {np.median(abs_errors):.2f}')
print(f'90th percentile error: {np.percentile(abs_errors, 90):.2f}')
print(f'Max error: {abs_errors.max():.2f}')

## 24. Interpretation and Business Insight

- **Relative Compactness** is the strongest predictor — more compact buildings need more heating
- **Overall Height** strongly affects heating load — taller buildings have higher loads
- **Surface Area** is inversely correlated — larger surface area = more heat loss
- **Glazing Area** has a moderate positive effect on heating (more glass = more heat loss)
- **Orientation** has minimal effect — building shape matters far more than compass direction
- These results align well with building physics principles

## 25. Limitations

1. Simulated data (Ecotect) — not real measured consumption
2. Only 768 samples with 12 unique building shapes
3. No climate/weather variables
4. No occupancy or usage patterns
5. Fixed material properties assumed in simulation

## 26. How to Improve This Project

1. Include real building energy data (e.g., ASHRAE Great Energy Predictor)
2. Add weather and climate variables
3. Multi-output regression (predict both Heating and Cooling Load)
4. Cross-validation for more reliable estimates on small data
5. Feature selection to identify minimal feature set

## 27. Production Considerations

- Building design optimization tool
- Integration with BIM (Building Information Modeling) software
- HVAC sizing calculator
- Energy compliance checking for building codes
- Climate-adjusted models for different regions

## 28. Common Mistakes

1. Not dropping Cooling_Load when predicting Heating_Load (leakage)
2. Using Orientation as a continuous variable (it's really categorical: N/S/E/W)
3. Not scaling features for linear models
4. Overfitting on 768 rows with complex models
5. Ignoring the simulation nature of the data

## 29. Mini Challenge / Exercises

1. Predict Cooling_Load instead and compare R²
2. Treat Orientation and Glazing_Area_Distribution as categorical
3. Use polynomial features and compare Ridge vs Lasso
4. Run 10-fold cross-validation and report mean ± std R²
5. Try a multi-output regression for both loads simultaneously

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict building heating load |
| Dataset | UCI Energy Efficiency (768 records) |
| Difficulty | Easy |
| Key insight | Compactness and height dominate energy consumption |
| Best approach | Gradient boosting on pure numeric features |
| Primary metric | R² |
| Baseline comparison | All models strongly beat mean baseline |